## Add Demodata

In [1]:
from dotenv import load_dotenv
import chromadb
from chromadb.config import Settings
from datasets import load_dataset, Dataset
from datasets import get_dataset_config_names
from mylibs.classes.AppSettings import AppSettings
settings = AppSettings()

c:\Projekte\OTOBO\OTOBO-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### german Q/A data for testing from HuggingFace

In [2]:
name_ds = "deepset/germanquad"
configs = get_dataset_config_names(name_ds)
print(configs)

demodata: Dataset = load_dataset(name_ds, split='test')  # type: ignore

print(demodata.num_rows)
print(demodata.shape) # Shape of the dataset (number of columns, number of rows).
print(demodata.column_names)
print(demodata.features)

['plain_text']
2204
(2204, 4)
['id', 'context', 'question', 'answers']
{'id': Value(dtype='int32', id=None), 'context': Value(dtype='string', id=None), 'question': Value(dtype='string', id=None), 'answers': Sequence(feature={'text': Value(dtype='string', id=None), 'answer_start': Value(dtype='int32', id=None)}, length=-1, id=None)}


### connect to Chroma

In [3]:
load_dotenv()

chroma_client = chromadb.HttpClient(host=settings.AI_VECTORDB_HOST, port=settings.AI_VECTORDB_PORT,settings=Settings(
            chroma_client_auth_provider="chromadb.auth.token.TokenAuthClientProvider",
            chroma_client_auth_credentials=settings.AI_VECTORDB_AUTH_TOKEN,
        ))

colls = chroma_client.list_collections()
colls

[Collection(name=ollama_embedding),
 Collection(name=my_collection),
 Collection(name=documents),
 Collection(name=openai_collection),
 Collection(name=testdata)]

In [4]:
# Achtung: danach sind die Collection Connections weg => Server neu starten
# chroma_client.delete_collection(settings.AI_VECTORSTORE_INDEX)

collection = chroma_client.get_or_create_collection(settings.AI_VECTORSTORE_INDEX)

In [7]:
collection.count()
# collection.name

695

In [11]:
where = {"process_id": "Slice_100_200"}
get_all_result = collection.get(where=where, include=["documents", "metadatas"])
get_all_result

{'ids': ['8966bbaa-92da-4e14-be05-a2429ebd6097',
  '76bd46dc-4f16-4867-b722-4f26369b61d8',
  'cd7cbb2c-3f3a-488a-a0d2-c81dc615b91d',
  'de5f7c3a-f925-4a75-a1c0-aea3226437e0',
  '84962fd3-480d-4d32-a444-e265dcc8ab83',
  'a60f4c13-09e6-4151-855c-d92cef5a47b7',
  'f41cd810-4df5-48e8-a2f7-b53aa00cb6d1',
  '27d7d49b-b79e-4887-b1ab-8b1d4381c409',
  'df89dc4c-d026-4c6c-b666-67771c828ed4',
  'b4ed0634-566f-4094-a0f3-3b95229c3ae3',
  'f21f329e-1697-4809-b244-0c981ce11f66',
  '8a373b83-27d7-4a81-8ac4-656dab9e664a',
  'c66dbd0e-9f11-49e0-a1f7-6d246f2ad624',
  '5a0cb9d6-2019-48dd-8bb0-18b9de208264',
  '5e2df028-c6a9-4dad-b749-72f128e86492',
  'b7d9818b-10e0-4662-8ef7-002b5cd65956',
  '0272a274-7423-4ec8-9b4b-257bd76a8558',
  'd9e44479-c221-4651-8ec9-ceb7bdf091b5',
  '772b4330-bb2b-438c-bcde-e6a69c173aea',
  'f49c01d6-8924-43c0-9a23-5be1e5fd9412',
  '83a08dda-63cc-4aea-935a-888f2830d873',
  'e59d323b-98d1-4753-a369-4d17f677b15b',
  '9f2c126c-727a-460a-a961-7689601b9cf7',
  'e6be8396-b787-479f-9160-

In [ ]:
collection.peek(100)


### slice into pieces of 50 

In [13]:
import json
# split in first 100 and rest
ab = 150
bis = 200 # dataset.num_rows

slice = demodata[ab:bis]
# !!! len(slice) is 4 because ['id', 'context', 'question', 'answers']

# slice = dataset[anz:]

data = []
for i in range(bis-ab):
  question = {}
  question["id"] = f'Q_{slice["id"][i]}'
  # question["process_id"]= str(slice["id"][i])
  question["process_id"]= f"Slice_{ab}_{bis}"
  question["gdpr_id"]= str(slice["id"][i])
  question["topic"]= slice["question"][i]
  question["type"]= "question"
  question["len"] = len(slice["question"][i])
  question["document"]= slice["question"][i]
  data.append(question)

  answer = {}
  answer["id"] = f'A_{slice["id"][i]}'
  # answer["process_id"]= str(slice["id"][i])
  answer["process_id"]= f"Slice_{ab}_{bis}"
  answer["gdpr_id"]= str(slice["id"][i])
  answer["topic"]= slice["answers"][i]["text"][0]
  answer["type"]= "answer"
  answer["len"] = len(slice["context"][i])
  answer["document"]= slice["context"][i]
  data.append(answer)


## Or

In [12]:
import json

n = 3

steps = 100

ab = n * steps
bis = (n+1) * steps

slice = demodata[ab:bis]
# !!! len(slice) is 4 because ['id', 'context', 'question', 'answers']

# slice = dataset[anz:]

datas = []
for i in range(bis-ab):
  data = {}
  data["id"] = slice["id"][i]
  data["process_id"]= f"Slice_{ab}_{bis}"
  data["gdpr_id"]= str(slice["id"][i])
  data["topic"]= slice["question"][i]
  data["type"]= "answer"

  document = f'Q: {slice["question"][i]}\nA: {slice["answers"][i]["text"][0]}\nTopic: {slice["context"][i]}'
  data["document"]= document
  data["len"] = len(document)

  datas.append(data)

In [13]:
len(datas)
# data

100

In [33]:
datas[0]["process_id"]

'Slice_0_10'

In [14]:
import os
import requests

api_url = "http://127.0.0.1:8000/ai/embedding/insert/"

headers = {
  'Content-Type': 'application/json',
  'access_token': settings.AI_API_KEY
}

response = requests.put(api_url, json=datas, headers=headers)
if response.status_code != 200:
  print("Error at: ", response.json)

In [ ]:
print(response.json())
len(response.json())

In [ ]:
import pandas as pd

df = pd.DataFrame(data)
print(df)